Groundwater | Transport Modeling Track

# Step 2: Perceptual Model – From Flow to Transport

> **The 10 steps at a glance:** 1-Goal → **2-Perceptual** → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 1, you defined clear objectives for the transport model: simulate heat transport, evaluate thermal impacts, and assess river–aquifer thermal exchange. Now we build a **perceptual model for transport** – what additional understanding do we need beyond the calibrated flow model?

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~40–50 min core + 20–25 min optional sections |
| **Prerequisites** | Flow track Steps 1–2, Transport track Step 1 |

**Learning objectives:**

By the end of this notebook you will be able to:

1. Calculate **groundwater velocity** from Darcy flux and effective porosity
2. Estimate **mechanical dispersion** and assess the Peclet number for the Limmat Valley
3. Derive **bulk thermal properties** and the thermal retardation factor
4. Identify **temperature boundary conditions** from available data
5. Construct a **thermal energy budget** for the model domain

In [ ]:
# Setup
import sys
import os

import pyproj
os.environ['PROJ_LIB'] = pyproj.datadir.get_data_dir()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file
from map_utils import display_concessions_map
import climate_utils as cu
from shared_functions import check_task_with_solution, create_multiple_choice

---
## 1 — From Flow to Transport

The calibrated flow model gives us the **velocity field** (specific discharge $q$). To simulate transport we need additional information:

| Flow model gave us | Transport adds |
|---|---|
| Hydraulic conductivity $K$ | Effective porosity $n_e$ → **velocity** $v = q / n_e$ |
| Hydraulic gradient $i$ | Dispersivity $\alpha_L$ → **mechanical dispersion** |
| Water balance (fluxes) | Temperature of each flux → **thermal boundary conditions** |
| River–aquifer exchange rates | Thermal properties of water and solids → **retardation** |

This notebook works through each of these additions.

---
## 2 — Groundwater Velocity

The **specific discharge** (Darcy flux) $q = Ki$ tells us the volume of water crossing a unit area per unit time, but it is not the speed at which water (or a tracer) actually moves through the pore space.

The **seepage velocity** (average linear velocity) is:

$$v = \frac{q}{n_e} = \frac{Ki}{n_e}$$

For the Limmat Valley glaciofluvial gravels, effective porosity $n_e$ is typically **0.15–0.30** (Freeze & Cherry 1979, Table 2.4). We adopt $n_e = 0.20$ as a reasonable central estimate for well-sorted gravel.

In [ ]:
# Representative values from the flow model
K = 0.01          # hydraulic conductivity [m/s]  (~864 m/d)
dh = 2.0          # head difference across domain [m]
dx = 780.0        # flow path length [m]
i = dh / dx       # hydraulic gradient [-]
n_e = 0.20        # effective porosity [-]

# Specific discharge
q = K * i  # [m/s]

# Seepage velocity
v = q / n_e  # [m/s]
v_m_per_day = v * 86400  # convert to m/day

print(f"Hydraulic gradient:     i = {i:.4f}")
print(f"Specific discharge:     q = {q:.2e} m/s  ({q * 86400:.2f} m/d)")
print(f"Seepage velocity:       v = {v:.2e} m/s  ({v_m_per_day:.1f} m/d)")

# Travel time across ~4.5 km domain
L_domain = 4500  # [m]
travel_time_days = L_domain / v_m_per_day
print(f"\nTravel time across {L_domain/1000:.1f} km domain: ~{travel_time_days:.0f} days")

**Key insight:** The seepage velocity (~11 m/day) is much faster than the specific discharge (~2 m/day). This distinction matters for:

- **Wellhead protection zones** — defined by travel time of water (velocity), not flux
- **Contaminant arrival times** — a spill 1 km upgradient could reach a well in ~90 days
- **Thermal plume propagation** — heat moves at $v / R$ where $R$ is the retardation factor (Section 4)

### Checkpoint 1 — Seepage Velocity

In [ ]:
check_task_with_solution('task_t02_checkpoint_1')

---
## 3 — Dispersion

In a porous medium, a solute (or thermal) front does not advance as a sharp plug — it **spreads** due to:

1. **Mechanical dispersion** — velocity variations at the pore scale cause differential advection
2. **Molecular diffusion** — random thermal motion of molecules (usually negligible in fast-flowing gravels)

The **longitudinal hydrodynamic dispersion coefficient** combines both:

$$D_L = \alpha_L \cdot v + D_m^*$$

where $\alpha_L$ is the longitudinal dispersivity [m] and $D_m^*$ is the effective molecular diffusion coefficient (~$10^{-9}$ m$^2$/s in porous media).

### Scale-dependent dispersivity

Dispersivity is notoriously **scale-dependent**: it increases with the distance over which transport is observed (Gelhar et al. 1992). For our model scale (~1–5 km):

| Observation scale | Typical $\alpha_L$ range | Reference |
|---|---|---|
| Laboratory column (0.1–1 m) | 0.001–0.01 m | Freeze & Cherry 1979 |
| Field tracer test (10–100 m) | 0.1–10 m | Gelhar et al. 1992 |
| Regional model (1–10 km) | 5–100 m | Gelhar et al. 1992 |

For the Limmat Valley at our model scale, $\alpha_L \approx$ **5–20 m** is a reasonable starting range. The transverse dispersivity $\alpha_T$ is typically 1/10 of $\alpha_L$.

In [ ]:
# Peclet number: advection vs. diffusion
alpha_L = 10.0         # longitudinal dispersivity [m]
D_m_star = 1e-9        # effective molecular diffusion [m²/s]

D_L = alpha_L * v + D_m_star  # longitudinal dispersion coefficient [m²/s]

# Peclet number: Pe = v * L / D_m*  (ratio of advective to diffusive transport)
L_char = 100  # characteristic length [m] (e.g., grid cell size)
Pe = v * L_char / D_m_star

print(f"Longitudinal dispersivity:  alpha_L = {alpha_L} m")
print(f"Dispersion coefficient:     D_L = {D_L:.2e} m²/s")
print(f"Molecular diffusion:        D_m* = {D_m_star:.0e} m²/s")
print(f"\nPeclet number (L={L_char} m):  Pe = {Pe:.0e}")
print(f"\n→ Pe >> 1: transport is strongly advection-dominated.")
print(f"  Mechanical dispersion ({alpha_L * v:.2e} m²/s) >> molecular diffusion ({D_m_star:.0e} m²/s)")

<details>
<summary><b>Optional: Grid Peclet criterion</b> (click to expand)</summary>

To avoid numerical oscillations in a finite-difference transport model, the **grid Peclet number** should satisfy:

$$Pe_{grid} = \frac{v \cdot \Delta x}{D_L} \lesssim 2$$

For our values ($v \approx 1.3 \times 10^{-4}$ m/s, $D_L \approx 1.3 \times 10^{-3}$ m$^2$/s):

$$\Delta x \lesssim \frac{2 \cdot D_L}{v} = \frac{2 \times 1.3 \times 10^{-3}}{1.3 \times 10^{-4}} \approx 20 \text{ m}$$

This means our transport grid should have cells no larger than ~20 m in the flow direction, or we need to use TVD (Total Variation Diminishing) schemes that are less sensitive to the grid Peclet constraint.

</details>

---
## 4 — Thermal Properties of the Aquifer

Heat transport requires thermal properties of both the **water** and the **solid matrix**. Unlike dispersivity, these are well-constrained for common geological materials:

| Property | Symbol | Water | Gravel/sand solids | Unit |
|---|---|---|---|---|
| Density | $\rho$ | 1000 | 2650 | kg/m$^3$ |
| Specific heat capacity | $c$ | 4184 | 880 | J/(kg K) |
| Thermal conductivity | $\lambda$ | 0.60 | 2.0 | W/(m K) |
| Volumetric heat capacity | $\rho c$ | 4.18 $\times 10^6$ | 2.33 $\times 10^6$ | J/(m$^3$ K) |

In [ ]:
# Thermal properties
rho_w = 1000.0    # water density [kg/m³]
c_w = 4184.0      # water specific heat capacity [J/(kg·K)]
rho_s = 2650.0    # solid density [kg/m³]
c_s = 880.0       # solid specific heat capacity [J/(kg·K)]
lambda_w = 0.60   # water thermal conductivity [W/(m·K)]
lambda_s = 2.0    # solid thermal conductivity [W/(m·K)]
n_e_thermal = 0.20  # porosity (same as above)

# Bulk thermal conductivity (geometric mean)
lambda_bulk = lambda_w**n_e_thermal * lambda_s**(1 - n_e_thermal)

# Bulk volumetric heat capacity
rho_c_bulk = n_e_thermal * rho_w * c_w + (1 - n_e_thermal) * rho_s * c_s

# Thermal retardation factor
# R = (rho*c)_bulk / (rho_w * c_w * n_e)
R_thermal = rho_c_bulk / (rho_w * c_w * n_e_thermal)

# Thermal front velocity
v_thermal = v / R_thermal  # [m/s]

print("Bulk thermal properties:")
print(f"  Thermal conductivity:     lambda_bulk = {lambda_bulk:.2f} W/(m·K)")
print(f"  Volumetric heat capacity: (rho·c)_bulk = {rho_c_bulk:.2e} J/(m³·K)")
print(f"\nThermal retardation factor: R = {R_thermal:.2f}")
print(f"  → A thermal front moves {R_thermal:.1f}x slower than the groundwater velocity")
print(f"  → Thermal front velocity: v_T = {v_thermal * 86400:.1f} m/day  (vs. v_water = {v_m_per_day:.1f} m/day)")

### Why heat is better constrained than solutes

| Aspect | Heat transport | Solute transport |
|---|---|---|
| Key uncertain parameter | Thermal conductivity $\lambda$ | Dispersivity $\alpha_L$ |
| Typical range | Factor of 2–3 | 1–3 orders of magnitude |
| Scale dependence | Weak | Strong (Gelhar 1992) |
| Additional process | Thermal retardation ($R \approx 2$–$4$) | Sorption (highly species-specific) |
| Observable | Temperature (cheap, continuous) | Concentration (expensive, sparse) |

This is why we focus on **heat transport** in this course: the physics is the same as solute transport, but the parameters are better constrained and the data is more readily available.

<details>
<summary><strong>For solute applications: retardation via sorption</strong></summary>

The thermal retardation factor $R \approx 3.5$ we just calculated is a **bulk property of the aquifer** — it depends only on porosity and the volumetric heat capacities of water and solids, which are well-constrained. Every heat signal is retarded the same way.

For solute transport, the analogous process is **sorption** — dissolved species adsorbing onto grain surfaces. The solute retardation factor is:

$$R_s = 1 + \frac{\rho_b \cdot K_d}{n_e}$$

where $\rho_b$ is the bulk density and $K_d$ is the distribution coefficient [L/kg].

The key differences:

| | Thermal retardation | Solute retardation |
|---|---|---|
| **Mechanism** | Heat exchange between water and solids | Adsorption onto grain surfaces |
| **Typical $R$** | 2–4 (narrow range) | 1–1000+ (species-dependent) |
| **Predictability** | High — bulk material property | Low — depends on species, pH, organic content |
| **Example** | All thermal fronts in gravel: $R \approx 3$–$4$ | Chloride (conservative): $R = 1$; heavy metals: $R > 100$ |

**What this means for modelling:** When we calibrate with heat, the retardation factor is essentially known — so the calibration constrains $n_e$ and $\alpha_L$ directly. For a solute application, $R_s$ becomes an additional unknown that must be estimated for each species.

</details>

### Checkpoint 2 — Thermal Retardation Factor

In [ ]:
check_task_with_solution('task_t02_checkpoint_2')

---
## 5 — Temperature in the Limmat Valley

To set up thermal boundary conditions, we need to know the **temperature** of each water flux entering or leaving the model domain. Let's review the available data.

### 5.1 — Air Temperature as Background Proxy

In shallow unconfined aquifers, the mean annual groundwater temperature is often close to the **mean annual air temperature** (MAAT), typically within $\pm$1–2 $^\circ$C. Let's retrieve the climate data for Zurich.

In [ ]:
# Download and plot climate data
climate_data_path, climate_norms = cu.get_and_plot_climate_data(
    station_string="Fluntern",
    custom_title="Climate Normals — Zurich Fluntern"
)

# Extract mean annual air temperature
if 'Year' in climate_norms.index:
    annual_row = climate_norms.loc['Year']
elif 'Annual' in climate_norms.index:
    annual_row = climate_norms.loc['Annual']
else:
    # Try last row as annual summary
    annual_row = climate_norms.iloc[-1]

# Find temperature column
temp_cols = [c for c in climate_norms.columns if 'temp' in c.lower() or 'T' == c.strip()]
if temp_cols:
    T_air = float(annual_row[temp_cols[0]])
else:
    T_air = 9.8  # MeteoSwiss published value

print(f"\nMean annual air temperature (Zurich Fluntern): {T_air:.1f} °C")
print(f"→ We use this as the background groundwater temperature proxy: T_gw ≈ {T_air:.0f}–{T_air+1:.0f} °C")

### 5.2 — River Temperatures

The two rivers in our model domain have distinctly different thermal regimes:

| River | Mean annual T | Source | Key influence |
|---|---|---|---|
| **Sihl** (at Sihlhölzli) | ~10.6 $^\circ$C | Canton of Zurich Water Yearbook | Alpine catchment, relatively natural |
| **Limmat** (at KW Letten) | ~12.5 $^\circ$C | Canton of Zurich Water Yearbook | Lake Zurich outflow, urban heat island |

**Why is the Limmat warmer?**
- Lake Zurich acts as a **thermal buffer** — it stores summer heat and releases it slowly
- The Limmat flows through the **urban core** of Zurich, absorbing waste heat
- Thermal effluents from cooling water returns add additional heat

The ~2 $^\circ$C difference between the Limmat and background groundwater temperature makes river–aquifer exchange a significant **thermal input** to the system.

In [ ]:
# Temperature reference values
T_air_ref = 9.8      # Mean annual air temperature [°C]
T_gw_background = 10.5  # Background GW temperature [°C] (slightly above air T)
T_sihl = 10.6        # River Sihl mean annual [°C]
T_limmat = 12.5      # River Limmat mean annual [°C]
T_recharge = T_air_ref  # Areal recharge temperature [°C]
T_lateral = T_gw_background  # Lateral inflow temperature [°C]

print("Temperature reference values for the Limmat Valley model:")
print(f"{'Component':<30} {'T [°C]':>8}  {'Source'}")
print("-" * 70)
print(f"{'Mean annual air T (Fluntern)':<30} {T_air_ref:>8.1f}  MeteoSwiss")
print(f"{'Background GW temperature':<30} {T_gw_background:>8.1f}  Air T + ~0.5–1 °C")
print(f"{'River Sihl (Sihlhölzli)':<30} {T_sihl:>8.1f}  Canton yearbook")
print(f"{'River Limmat (KW Letten)':<30} {T_limmat:>8.1f}  Canton yearbook")
print(f"{'Areal recharge':<30} {T_recharge:>8.1f}  ≈ air temperature")
print(f"{'Lateral inflow from hills':<30} {T_lateral:>8.1f}  ≈ background GW")

### 5.3 — Groundwater Quality Monitoring

The Canton of Zurich operates **groundwater quality monitoring stations** that include temperature measurements. However, for our model area specifically:

- **3 monitoring stations** (Hardturm, Letten, Stampfenbach) lie just **outside** or at the edge of our model boundary
- **No continuous temperature monitoring** exists inside the model domain itself
- Published mean GW temperatures at these stations range from **10–13 $^\circ$C**

This means our temperature boundary conditions carry an uncertainty of approximately **$\pm$2 $^\circ$C** — significantly better than the uncertainty on dispersivity, but still relevant for thermal impact assessments.

> Data source: [Canton of Zurich Groundwater Yearbook](https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/grundwasser.html)

### 5.4 — Thermal Concessions (Heat Pump and Cooling Water Use)

The Limmat Valley is one of Switzerland's most intensely used aquifers for **thermal energy**. Groundwater heat pumps (WPG = *Wärmepumpe Grundwasser*) and cooling water systems (KW = *Kühlwasser*) extract groundwater, exchange heat, and reinject it — creating local temperature anomalies of $\pm$3–5 $^\circ$C.

Let's map the thermal concessions in our model area.

In [ ]:
# Download well and boundary data
wells_path = download_named_file(name='wells', data_type='gis')
model_boundary_path = download_named_file(name='model_boundary', data_type='gis')

# Read and clip to model boundary
gdf_boundary = gpd.read_file(model_boundary_path)
wells_gdf = gpd.read_file(wells_path, layer='GS_GRUNDWASSERFASSUNGEN_OGD_P')
wells_gdf = wells_gdf.clip(gdf_boundary)

# Add concession ID
wells_gdf['concession_id'] = wells_gdf['GWR_ID'].str.split('_').str[0]

# Filter active wells
is_decommissioned = wells_gdf['BESCHREIBUNG'].str.contains('aufgehoben', case=False, na=False)
is_unused = wells_gdf['BESCHREIBUNG'].str.contains('ungenutzt', case=False, na=False)
wells_active = wells_gdf[~is_decommissioned & ~is_unused].copy()

# Filter for thermal use: WPG (heat pump) or KW (cooling water)
is_thermal = wells_active['NUTZART'].str.contains('WPG|KW', case=False, na=False)
thermal_wells = wells_active[is_thermal].copy()

n_thermal_concessions = thermal_wells['concession_id'].nunique()
n_all_concessions = wells_active['concession_id'].nunique()

print(f"Active concessions in model area: {n_all_concessions}")
print(f"Thermal concessions (WPG/KW):     {n_thermal_concessions}")
print(f"Fraction thermal:                 {n_thermal_concessions/n_all_concessions:.0%}")

In [ ]:
# Map of thermal concessions
display_concessions_map(
    thermal_wells,
    boundary_gdf=gdf_boundary,
    map_title="Thermal Groundwater Concessions (WPG/KW) — Limmat Valley"
)

**Observations from the map:**

- Thermal wells are **concentrated in the city centre** (Zurich districts 1–5), reflecting high heating/cooling demand
- Individual concessions are not modelled in our base case — their aggregate effect is part of the background thermal state
- In the **case study** (Notebook 8), students will add specific thermal doublets and investigate thermal plume propagation

### Checkpoint 3 — Thermal Well Distribution

In [ ]:
create_multiple_choice('task_t02_checkpoint_3')

---
## 6 — Thermal Energy Budget

Each water flux carries thermal energy. The **heat flux** associated with a volumetric water flux $Q$ at temperature $T$ is:

$$\Phi = \rho_w \cdot c_w \cdot Q \cdot T$$

More usefully, the **temperature anomaly** $\Delta T = T_{flux} - T_{background}$ tells us whether a flux is warming or cooling the aquifer relative to background conditions.

Let's assemble the energy budget using the water balance from the flow perceptual model (Notebook 2) and the temperatures we've established.

In [ ]:
# Water fluxes from flow perceptual model (10^6 m³/yr)
# Positive = into domain, Negative = out of domain
T_bg = T_gw_background  # 10.5 °C reference

energy_budget = pd.DataFrame({
    "Component": [
        "Areal recharge",
        "Lateral inflow (south)",
        "Lateral inflow (north)",
        "River Sihl infiltration",
        "River Limmat / Hardhof recharge",
        "Hardhof pumping (out)",
        "GW outflow (west)",
    ],
    "Q [10^6 m³/yr]": [1.1, 1.5, 1.0, 1.5, 7.7, -7.0, -4.8],
    "T [°C]": [
        T_recharge,     # 9.8 — air temperature
        T_lateral,      # 10.5 — background GW
        T_lateral,      # 10.5 — background GW
        T_sihl,         # 10.6 — slightly above background
        T_limmat,       # 12.5 — significantly warmer
        T_gw_background,  # 10.5 — extracted at local GW temp
        T_gw_background,  # 10.5 — outflow at background
    ],
})

# Temperature anomaly relative to background
energy_budget["Delta_T [°C]"] = energy_budget["T [°C]"] - T_bg

# Thermal power anomaly: Phi = rho_w * c_w * Q * Delta_T  [W]
# Q in 10^6 m³/yr → m³/s: multiply by 1e6 / (365.25 * 86400)
Q_m3_per_s = energy_budget["Q [10^6 m³/yr]"] * 1e6 / (365.25 * 86400)
energy_budget["Phi_anomaly [kW]"] = (rho_w * c_w * Q_m3_per_s * energy_budget["Delta_T [°C]"]) / 1000

# Display
print("Thermal Energy Budget — Limmat Valley Aquifer")
print(f"Reference background temperature: T_bg = {T_bg} °C\n")
print(energy_budget.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

print(f"\nNet thermal anomaly input: {energy_budget['Phi_anomaly [kW]'].sum():.0f} kW")

**Interpretation:**

| Component | $\Delta T$ | Effect | Significance |
|---|---|---|---|
| Areal recharge | $-$0.7 $^\circ$C | Slight cooling | Small flux, small anomaly |
| Lateral inflows | 0.0 $^\circ$C | Neutral | At background temperature |
| River Sihl | +0.1 $^\circ$C | Near-neutral | Close to background |
| **River Limmat / Hardhof** | **+2.0 $^\circ$C** | **Warming** | **Dominant thermal input** — large flux at elevated T |
| Outflows | 0.0 $^\circ$C | Carry away heat at local T | Thermal equilibrium pathway |

The **Limmat / Hardhof recharge** is by far the dominant source of thermal energy to the aquifer. This river filtrate at ~12.5 $^\circ$C enters the aquifer 2 $^\circ$C warmer than background, creating a persistent thermal plume that extends downstream along the flow direction.

### Checkpoint 4 — Dominant Thermal Input

In [ ]:
create_multiple_choice('task_t02_checkpoint_4')

---
## 7 — Could We Be Wrong?

Every perceptual model involves simplifications. Here are the main **conceptual alternatives** we should keep in mind:

| Assumption | Alternative | Impact if wrong |
|---|---|---|
| Thermal equilibrium between water and solids (instantaneous) | Finite heat exchange rate | R would be smaller; thermal front moves faster |
| 2D horizontal (depth-averaged temperature) | 3D with vertical temperature gradients | Underestimate near-surface thermal impacts |
| Steady-state flow field (from flow model) | Transient flow (seasonal water table fluctuations) | Seasonal thermal plume dynamics missed |
| Uniform porosity $n_e = 0.20$ | Spatially variable $n_e$ (0.15–0.30) | Local velocity variations, different travel times |
| Background T = 10.5 $^\circ$C everywhere | Spatial gradient in background T | Would shift baseline for thermal impact assessment |
| No thermal concessions in base model | Include individual heat pump doublets | Urban heat island effect underestimated |

The most consequential assumption for our purposes is using a **steady-state flow field** for transient transport. This is justified for long-term average conditions but means we cannot capture seasonal thermal dynamics (e.g., summer warming pulses from the Limmat).

---
## What You're Taking Forward

| Parameter | Value | Unit | Source |
|---|---|---|---|
| Effective porosity $n_e$ | 0.20 | — | Literature (glaciofluvial gravel) |
| Seepage velocity $v$ | ~11 | m/day | $v = Ki/n_e$ |
| Longitudinal dispersivity $\alpha_L$ | 5–20 | m | Gelhar et al. 1992 |
| Thermal retardation $R$ | ~3.5 | — | $R = (\rho c)_{bulk} / (n_e \rho_w c_w)$ |
| Thermal front velocity | ~3 | m/day | $v_T = v / R$ |
| Background GW temperature | 10.5 | $^\circ$C | Air T + 0.5–1 $^\circ$C |
| River Sihl temperature | 10.6 | $^\circ$C | Canton yearbook |
| River Limmat temperature | 12.5 | $^\circ$C | Canton yearbook |
| Dominant thermal input | Limmat / Hardhof | — | Energy budget analysis |

> **Next step:** In [Step 3 (MODFLOW for Transport)](./3_modflow_transport.ipynb), we translate this perceptual model into a MODFLOW 6 GWT (Groundwater Transport) model configuration.

---
## References

- Anderson, M.P. (2005). *Heat as a ground water tracer*. Ground Water, 43(6), 951–968.
- Doppler, T. et al. (2007). *Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions*. Journal of Hydrology, 347(1–2), 177–187.
- Fetter, C.W. (2001). *Applied Hydrogeology*. 4th ed. Prentice Hall.
- Freeze, R.A. & Cherry, J.A. (1979). *Groundwater*. Prentice Hall.
- Gelhar, L.W., Welty, C. & Rehfeldt, K.R. (1992). *A critical review of data on field-scale dispersion in aquifers*. Water Resources Research, 28(7), 1955–1974.
- Canton of Zurich (2023). *Gewässer- und Grundwasser-Jahrbuch*. AWEL.